# MLR (Ridge) — SDY569 federated with the other institutions (SDY569's view)

**Stage 2.** The per-study notebooks (Stage 1) fit each institution on the four
LASSO-selected features (weight_kg, GAD65, received_active_treatment, Sex) and
wrote a model artifact. This notebook reads SDY569's partners'
artifacts (SDY524) and applies them to **SDY569's own
data** — showing what SDY569 gains by federating, without any subject-level data
leaving any institution.

**Solo**: SDY569 predicts its held-out subjects with its own model.
**Federated**: SDY569 combines its model with the partners' (read from disk) and
predicts the same subjects. We report R², MSE, and achieved power for both, and
the scatter of observed vs predicted.


## 1. Setup

In [ ]:
from __future__ import annotations
import sys, os, warnings, pickle
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

REPO = Path.cwd()
if (REPO / "src").exists():
    sys.path.insert(0, str(REPO / "src"))
elif (REPO.parent / "src").exists():
    REPO = REPO.parent
    sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge

import oadr_data as od

RNG_SEED = 42
N_BOOT = 2000
SEL = ["weight_kg", "GAD65", "received_active_treatment", "Sex"]
THIS = "SDY569"
PARTNERS = ['SDY524']
np.random.seed(RNG_SEED)
(REPO / "figures").mkdir(exist_ok=True)
print("Repo root:", REPO, "| THIS =", THIS, "| partners =", PARTNERS)


## 2. Load SDY569's four-feature data and the partners' artifacts

SDY569's data is loaded locally. The partners' contributions are read from
disk — coefficient vectors (`vectors/*_ridge_sel_vector.csv`) — never their subjects.


In [ ]:
def load4(s):
    b = od.load_panel_b(s)
    for c in ("bmi", "height_cm", "weight_kg"):
        b[c] = b[c].fillna(b[c].median())
    bad = b["height_cm"] <= 0
    b.loc[bad, "height_cm"] = np.sqrt(b.loc[bad, "weight_kg"] / b.loc[bad, "bmi"]) * 100
    X, y, _ = od.panel_b_design_matrix(b)
    return X.reindex(columns=SEL).values.astype(float), y.values.astype(float)

X4, y4 = load4(THIS)
print(f"{THIS}: N={len(y4)}, features={SEL}")

def read_partner_vec(s):
    df = pd.read_csv(REPO / "vectors" / f"{s}_ridge_sel_vector.csv").set_index("feature")
    coef = df.reindex(SEL)["coefficient"].fillna(0.0).values.astype(float)
    return coef, float(df.loc["__intercept__", "coefficient"]), int(df.loc[SEL[0], "n_subjects"])

partner_models = {s: read_partner_vec(s) for s in PARTNERS}
for s, (c, b0, n) in partner_models.items():
    print(f"  partner {s} (N={n}): " + ", ".join(f"{f}={v:+.3f}" for f, v in zip(SEL, c)))


## 3. Solo vs federated — 5-fold cross-validation on SDY569

SDY569 holds out each fold, fits its own model on the rest, and predicts the
held-out subjects two ways: with its own model (**solo**) and with its model
combined with the partners' (sample-size-weighted mean of coefficients, **federated**). We report MSE and R²
with a **bootstrap 95% confidence interval** — important here because SDY569 has
few subjects, so the point estimate alone overstates precision.

A note on interpretation: the federated number is a **transfer** result. With
only SDY569's own subjects to score on, "federated" means *the larger study's
model applied to SDY569's subjects* — it shows the big study's coefficients fit
the small study, not that more N raised statistical power within SDY569.


In [ ]:
def metrics(y, p):
    mask = ~np.isnan(p)
    yy, pp = y[mask], p[mask]
    mse = np.mean((yy - pp) ** 2)
    rss = np.sum((yy - pp) ** 2); tss = np.sum((yy - yy.mean()) ** 2)
    r2 = 1 - rss / tss if tss > 0 else float("nan")
    return mse, r2

def boot_r2_ci(y, p, n_boot=N_BOOT, seed=RNG_SEED):
    mask = ~np.isnan(p); yy, pp = y[mask], p[mask]
    rng = np.random.default_rng(seed); n = len(yy); out = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        ys, ps = yy[idx], pp[idx]
        tss = np.sum((ys - ys.mean()) ** 2)
        out.append(1 - np.sum((ys - ps) ** 2) / tss if tss > 0 else np.nan)
    return float(np.nanpercentile(out, 2.5)), float(np.nanpercentile(out, 97.5))

kf = KFold(min(5, max(2, len(y4) // 2)), shuffle=True, random_state=RNG_SEED)
solo = np.full(len(y4), np.nan)
fed = np.full(len(y4), np.nan)
for tr, te in kf.split(X4):
    sc = MinMaxScaler().fit(X4[tr])
    model = Ridge(alpha=1.0).fit(sc.transform(X4[tr]), y4[tr])
    solo[te] = model.predict(sc.transform(X4[te]))
    own = (model.coef_, model.intercept_, len(tr))
    coefs = np.stack([own[0]] + [partner_models[s][0] for s in PARTNERS])
    ints = np.array([own[1]] + [partner_models[s][1] for s in PARTNERS])
    sizes = np.array([own[2]] + [partner_models[s][2] for s in PARTNERS])
    ac = np.average(coefs, axis=0, weights=sizes); ai = np.average(ints, weights=sizes)
    fed[te] = sc.transform(X4[te]) @ ac + ai

mse_s, r2_s = metrics(y4, solo); ci_s = boot_r2_ci(y4, solo)
mse_f, r2_f = metrics(y4, fed);  ci_f = boot_r2_ci(y4, fed)
print(f"{THIS} — MLR (Ridge) on the four selected features  (N={len(y4)})")
print(f"  solo      MSE={mse_s:.3f}  R2={r2_s:+.3f}  95% CI [{ci_s[0]:+.2f}, {ci_s[1]:+.2f}]")
print(f"  federated MSE={mse_f:.3f}  R2={r2_f:+.3f}  95% CI [{ci_f[0]:+.2f}, {ci_f[1]:+.2f}]")


## 4. Presentation graphic — SDY569 solo vs federated


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), constrained_layout=True)
lo = min(y4.min(), np.nanmin(solo), np.nanmin(fed))
hi = max(y4.max(), np.nanmax(solo), np.nanmax(fed))
panels = [(solo, THIS + " alone", mse_s, r2_s, ci_s),
          (fed, THIS + " + " + "+".join(PARTNERS), mse_f, r2_f, ci_f)]
for ax, (pred, title, mse, r2, ci) in zip(axes, panels):
    ax.scatter(y4, pred, c="#1f77b4", s=65, edgecolor="white")
    ax.plot([lo, hi], [lo, hi], "k--", alpha=0.4)
    ax.set_xlabel("Observed log(C-peptide AUC)")
    ax.set_ylabel("Predicted log(C-peptide AUC)")
    ax.set_title(f"{title}\nR2={r2:+.2f} [{ci[0]:+.2f}, {ci[1]:+.2f}]  MSE={mse:.3f}", fontweight="bold")
    ax.grid(alpha=0.25)
fig.suptitle("MLR (Ridge) — federation from " + THIS + "'s view (four selected features)",
             fontsize=13, fontweight="bold")
fig.savefig(f"figures/Ridge_{THIS}_federated.pdf", dpi=300)
fig.savefig(f"figures/Ridge_{THIS}_federated.png", dpi=220)
plt.show()


## 5. Output

`figures/Ridge_SDY569_federated.pdf` / `.png` — SDY569 solo vs
federated on the four LASSO-selected features, with R² (95% bootstrap CI) and
MSE. The partner coefficients were
read from disk; no subject-level data crossed institutions.

Read the confidence interval before the point estimate: with SDY569's small N
the CI is wide, and the federated result is a **transfer** of the larger study's
model onto SDY569's subjects — not a within-SDY569 gain in statistical power.
A separate a-priori power analysis (detectable effect vs N) is the right place
for power claims; we do not compute post-hoc power from the observed R² here.
